# 1. Graph 분석

In [1]:
import os
import sys
sys.path.append(os.path.abspath('../src'))

import re
import ast
import json
import networkx as nx

from utils.graph_visualizer import GraphVisualizer
from modules.builders.graph_builder import HeteroGraphBuilder

/home/hyeonjin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hyeonjin/miniconda3/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [2]:
visualizer = GraphVisualizer(output_dir="./logs/visualizations")
graph_builder = HeteroGraphBuilder(db_dir="./data/raw/BIRD_dev/dev_databases") # DB 경로

In [ ]:
def graph_visualizer(log_file_path, target_query_id, exp_name):
    if exp_name.startswith("experiment"):
        score_path = f"/home/hyeonjin/thesis_refactored/outputs/experiments/{exp_name}/score_analysis_{exp_name}.jsonl"
    elif exp_name.startswith("baseline"):
        score_path = f"/home/hyeonjin/thesis_refactored/outputs/baselines/{exp_name}/score_analysis_{exp_name}.jsonl"

    parsed_data = {
        'question': '', 'metadata': {}, 'seeds': [], 'extracted_nodes': [], 'final_nodes': [],
        'node_scores': {}, 'generated_sql': '', 'gold_schema': []
    }

    try:
        with open(score_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                if data.get("query_id") == target_query_id:
                    node_name = data['node_name']
                    parsed_data['node_scores'][node_name] = data['score']
                    if data['is_gold']:
                        parsed_data['gold_schema'].append(node_name)
        
        print(f"✅ score_analysis 파싱 완료: 점수 {len(parsed_data['node_scores'])}개, 정답(Gold) {len(parsed_data['gold_schema'])}개")
    except FileNotFoundError:
        print(f"❌ score_analysis 파일을 찾을 수 없습니다: {score_path}")

    # ==========================================================
    # 3. .log 파일 파싱 (질문, 메타데이터, 예측 노드, 예측 SQL 추출)
    # ==========================================================
    is_target_query = False
    in_metadata_block = False
    in_subgraph_block = False
    in_sql_block = False
    metadata_str = ""
    subgraph_str = ""

    question_start_pattern = re.compile(rf"Question\s+{target_query_id}:")
    question_any_pattern = re.compile(r"Question\s+\d+:")
    log_time_pattern = re.compile(r"^\[20\d{2}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\]")

    def _flatten_subgraph_dict(sg):
        """{'tbl': ['col1', ...], ...} → ['tbl', 'tbl.col1', ...] (FK 노드는 그대로 유지)"""
        flat = []
        for tbl, cols in sg.items():
            flat.append(tbl)
            for c in cols:
                if "->" in str(c) or "." in str(c):
                    flat.append(str(c))  # 이미 fully qualified
                else:
                    flat.append(f"{tbl}.{c}")
        return flat

    try:
        with open(log_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if question_start_pattern.search(line):
                    is_target_query = True
                    parsed_data['question'] = line.split(f"Question {target_query_id}:")[-1].strip()
                    continue
                    
                if is_target_query and question_any_pattern.search(line) and not question_start_pattern.search(line):
                    break
                if not is_target_query:
                    continue

                if in_sql_block:
                    if log_time_pattern.match(line):
                        in_sql_block = False # 새로운 로그 줄이 시작되면 캡처 종료
                    else:
                        # 줄바꿈을 유지하면서 SQL 이어 붙이기
                        parsed_data['generated_sql'] += "\n" + line.strip()
                        continue

                if "metadata: {" in line:
                    metadata_str = line.split("metadata: ")[1].strip()
                    if metadata_str.count('{') == metadata_str.count('}') and metadata_str.count('[') == metadata_str.count(']'):
                        try: parsed_data['metadata'] = ast.literal_eval(metadata_str)
                        except: pass
                    else:
                        in_metadata_block = True
                elif in_metadata_block:
                    metadata_str += line.strip()
                    if metadata_str.count('{') == metadata_str.count('}') and metadata_str.count('[') == metadata_str.count(']'):
                        in_metadata_block = False
                        try: parsed_data['metadata'] = ast.literal_eval(metadata_str)
                        except: pass

                # PCST(Extractor) 결과: subgraph_dict 파싱 (multi-line tolerant)
                if "subgraph_dict: {" in line:
                    subgraph_str = line.split("subgraph_dict: ")[1].strip()
                    if subgraph_str.count('{') == subgraph_str.count('}') and subgraph_str.count('[') == subgraph_str.count(']'):
                        try:
                            sg = ast.literal_eval(subgraph_str)
                            parsed_data['extracted_nodes'] = _flatten_subgraph_dict(sg)
                        except: pass
                    else:
                        in_subgraph_block = True
                elif in_subgraph_block:
                    subgraph_str += line.strip()
                    if subgraph_str.count('{') == subgraph_str.count('}') and subgraph_str.count('[') == subgraph_str.count(']'):
                        in_subgraph_block = False
                        try:
                            sg = ast.literal_eval(subgraph_str)
                            parsed_data['extracted_nodes'] = _flatten_subgraph_dict(sg)
                        except: pass
                    
                if "seeds: [" in line:
                    try: parsed_data['seeds'] = ast.literal_eval(line.split("seeds: ")[1].strip())
                    except: pass
                    
                if "Final Nodes: [" in line:
                    try: parsed_data['final_nodes'] = ast.literal_eval(line.split("Final Nodes: ")[1].strip())
                    except: pass
                    
                if "Generated SQL:" in line:
                    parsed_data['generated_sql'] = line.split("Generated SQL:")[1].strip()
                    in_sql_block = True
                    
    except FileNotFoundError:
        print(f"❌ 로그 파일을 찾을 수 없습니다: {log_file_path}")

    # ==========================================================
    # 5. NetworkX 그래프 재건 및 텍스트 변환
    # ==========================================================
    nx_graph = nx.Graph()
    node_meta = parsed_data.get('metadata', {}).get('node_metadata', {})
    edges = parsed_data.get('metadata', {}).get('edges', [])
    edge_types = parsed_data.get('metadata', {}).get('edge_types', [])

    if node_meta:
        for idx, name in node_meta.items():
            idx = int(idx)
            n_type = "table" if "." not in name and "->" not in name else "column"
            score = round(parsed_data['node_scores'].get(name, 0.0), 4)
            nx_graph.add_node(name, name=name, type=n_type, similarity_score=score)
            
        for i, (u_idx, v_idx) in enumerate(edges):
            u_name, v_name = node_meta.get(u_idx), node_meta.get(v_idx)
            if u_name and v_name:
                nx_graph.add_edge(u_name, v_name, type=edge_types[i] if i < len(edge_types) else "relation")

    seeds_text = [node_meta.get(idx, str(idx)) for idx in parsed_data['seeds']] if parsed_data['seeds'] else []

    # ==========================================================
    # 6. 📊 깔끔한 종합 진단 리포트 출력 (여기가 핵심!)
    # ==========================================================
    print("="*60)
    print(f"✅ 파싱 및 진단 완료: Question {target_query_id}")
    print("="*60)
    print(f"📝 Question     : {parsed_data['question']}")
    print(f"🤖 Generated SQL: {parsed_data['generated_sql']}")
    print("-" * 60)
    print(f"📊 Graph Size   : {nx_graph.number_of_nodes()} Nodes / {nx_graph.number_of_edges()} Edges")
    print(f"🏆 Gold Nodes     ({len(parsed_data['gold_schema'])}개): {parsed_data['gold_schema']}")
    print(f"🌱 Seed Nodes     ({len(seeds_text)}개): {seeds_text}")
    print(f"🔷 Extracted (PCST) ({len(parsed_data['extracted_nodes'])}개): {parsed_data['extracted_nodes']}")
    print(f"🏁 Final Nodes    ({len(parsed_data['final_nodes'])}개): {parsed_data['final_nodes']}")
    print("="*60)

    # ==========================================================
    # 7. HTML 렌더링
    # ==========================================================
    if nx_graph.number_of_nodes() > 0:
        html_file_name = f"analysis_{exp_name}_q{target_query_id}.html"
        visualizer.visualize(
            graph=nx_graph,
            question=parsed_data['question'],
            seeds=seeds_text,
            extracted_nodes=parsed_data['extracted_nodes'],
            final_nodes=parsed_data['final_nodes'],
            gold_nodes=parsed_data['gold_schema'], 
            file_name=html_file_name
        )
        print(f"HTML 파일이 {html_file_name}으로 저장되었습니다.")
    else:
        print("❌ 그래프 메타데이터가 파싱되지 않아 시각화를 건너뜁니다.")

In [11]:
log_file_path = "/home/hyeonjin/thesis_refactored/logs/baselines/baseline_g_retriever/baseline_g_retriever_20260328_022118.log"
target_query_id = 36
exp_name = "baseline_g_retriever"

In [ ]:
graph_visualizer(log_file_path=log_file_path, target_query_id=target_query_id, exp_name=exp_name)

✅ score_analysis 파싱 완료: 점수 94개, 정답(Gold) 11개
✅ 파싱 및 진단 완료: Question 36
📝 Question     : Under whose administration is the school with the highest number of students scoring 1500 or more on the SAT? Indicate their full names.
🤖 Generated SQL: SELECT
s.AdmFName1,
s.AdmLName1,
s.AdmFName2,
s.AdmLName2,
s.AdmFName3,
s.AdmLName3
FROM satscores ss
JOIN schools s ON ss.cds = s.CDSCode
WHERE ss.NumGE1500 = (
SELECT MAX(NumGE1500)
FROM satscores
)
LIMIT 1;
------------------------------------------------------------
📊 Graph Size   : 94 Nodes / 93 Edges
🏆 Gold Nodes   (11개): ['satscores', 'schools', 'satscores.cds', 'satscores.NumGE1500', 'schools.CDSCode', 'schools.AdmFName1', 'schools.AdmLName1', 'schools.AdmFName2', 'schools.AdmLName2', 'schools.AdmFName3', 'schools.AdmLName3']
🌱 Seed Nodes   (94개): ['schools', 'schools.District', 'schools.School', 'schools.CharterNum', 'satscores.enroll12', 'schools.Charter', 'schools.EdOpsName', 'satscores', 'schools.EILName', 'schools.AdmFName1', 'schools.

: 